In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 04 — Statistical Analysis\n",
    "**Stage 6: Are the findings statistically robust?**\n",
    "\n",
    "Tests:\n",
    "1. Spearman rho: defensive rank vs final position\n",
    "2. Pearson r: goals_against vs points\n",
    "3. Logistic regression: can defence predict Top 4?"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null, "metadata": {}, "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.insert(0, '..')\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "from scipy import stats\n",
    "from sklearn.linear_model import LogisticRegression\n",
    "from sklearn.model_selection import cross_val_score\n",
    "from sklearn.preprocessing import StandardScaler\n",
    "from sklearn.pipeline import Pipeline\n",
    "from IPython.display import display\n",
    "from config import DATA_PROC_DIR\n",
    "\n",
    "df = pd.read_csv(DATA_PROC_DIR / 'standings_features.csv')\n",
    "summary_df = pd.read_csv(DATA_PROC_DIR / 'hypothesis_summary.csv')\n",
    "print(f'Loaded {len(df)} rows')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null, "metadata": {}, "outputs": [],
   "source": [
    "# ============================================================\n",
    "# TEST 1: Spearman Rank Correlation per league\n",
    "# ============================================================\n",
    "print('SPEARMAN: Defensive Rank vs Final Position')\n",
    "print('=' * 50)\n",
    "\n",
    "spearman_results = []\n",
    "for league in df['league_key'].unique():\n",
    "    sub = df[df['league_key'] == league].dropna(\n",
    "        subset=['defensive_rank', 'position']\n",
    "    )\n",
    "    rho, p = stats.spearmanr(\n",
    "        sub['defensive_rank'].astype(float),\n",
    "        sub['position'].astype(float)\n",
    "    )\n",
    "    sig = 'p<0.05 significant' if p < 0.05 else 'not significant'\n",
    "    spearman_results.append({\n",
    "        'League': league, 'rho': round(rho, 3),\n",
    "        'p_value': round(p, 4), 'n': len(sub), 'Result': sig\n",
    "    })\n",
    "    print(f'  {league:12s}: rho={rho:.3f}, p={p:.4f} -> {sig}')\n",
    "\n",
    "display(pd.DataFrame(spearman_results))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null, "metadata": {}, "outputs": [],
   "source": [
    "# ============================================================\n",
    "# TEST 2: Pearson Correlation — GA vs Points\n",
    "# ============================================================\n",
    "clean = df.dropna(subset=['goals_against', 'points'])\n",
    "r, p  = stats.pearsonr(\n",
    "    clean['goals_against'].astype(float),\n",
    "    clean['points'].astype(float)\n",
    ")\n",
    "print(f'Pearson r (Goals Against vs Points):')\n",
    "print(f'  r  = {r:.4f}')\n",
    "print(f'  r2 = {r**2:.4f}')\n",
    "print(f'  p  = {p:.2e}')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null, "metadata": {}, "outputs": [],
   "source": [
    "# ============================================================\n",
    "# TEST 3: Logistic Regression — Predict Top 4\n",
    "# ============================================================\n",
    "feature_cols = [\n",
    "    'goals_against', 'goals_against_avg',\n",
    "    'defensive_rank', 'ga_vs_median'\n",
    "]\n",
    "model_data = df[feature_cols + ['is_top4']].dropna()\n",
    "X = model_data[feature_cols]\n",
    "y = model_data['is_top4'].astype(int)\n",
    "\n",
    "pipeline = Pipeline([\n",
    "    ('scaler', StandardScaler()),\n",
    "    ('clf', LogisticRegression(max_iter=1000, random_state=42))\n",
    "])\n",
    "\n",
    "cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')\n",
    "\n",
    "print(f'Logistic Regression: Predict Top 4 from Defence')\n",
    "print(f'5-fold CV Accuracy: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}')\n",
    "print(f'Baseline (majority class): {max(y.mean(), 1-y.mean()):.3f}')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
  "language_info": {"name": "python", "version": "3.10.0"}
 },
 "nbformat": 4,
 "nbformat_minor": 4
}